# Gemma 4 Agent: Budget-Fit Single Agent

A single agent on the official starter prompt, with the **time and tool-call budget chosen from measured runs** instead of guessed.

**What is different from the starter**

| Setting | Starter / common forks | Here | Why |
|---|---|---|---|
| Time per task | 4 min, or no cap | **8 min** | In 129 local runs, only about half of the fixes Gemma eventually finds are in the file after 4 minutes of scorer-speed time |
| Tool calls per task | 40-100 | **28** | Runs that are still exploring after many calls almost never succeed; a hard call cap stops them and pays for the longer time cap |
| Explore rule (prompt) | none | edit by the 12th call | Runs with no source edit by call 12 fail 87% of the time |
| `max_output_tokens` | 16384 | **8192** | vLLM refuses any request with prompt + max output above 32,768 tokens, which kills long tasks |
| Command timeout (`timeout_seconds`) | 60-120 s | **240 s** | In `swegemma/harness/verification.py` the scoring pytest run uses this same value as its timeout, and it runs whole test files. Locally the `requests` test files need about 100 s, so a correct patch can be scored as a failure with a short timeout. Agent commands rarely hit the limit (2 of about 3,700 calls in my runs) |
| Graph tools | recommended by the prompt | discouraged | They only match exact fully-qualified ids and usually return nothing |

The hidden test set runs about 120 tasks one after another within 12 hours, so every minute spent on a task that will fail is a minute taken from one that could pass. The caps above keep the total inside the limit: a scored run with these caps finished in under 12 hours.

**How to use:** CPU, Internet off, Run All, then submit this notebook's `submission.zip`.

Prompt base: the competition's official `sample_submission` prompt. Graph-tool behaviour: community audits in the discussion forum.

Scores on the 58-task public leaderboard move by about ±2 tasks between identical submissions, so one score is not a comparison.


In [ ]:
import zipfile
from pathlib import Path

FILES = {
 "agent.yaml": "name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n",
 "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n",
 "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n",
 "prompts/system.md": "You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.\n\n## Core Objective: Fast, Minimal, and Precise Fixes\nAim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8\u201310 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.\n\n## Workflow\n\n### 1. Identify Target Files Immediately\n- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.\n- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.\n- If the problem statement does not provide explicit file paths, search for its most specific identifier or error string with `git grep -n \"text\" -- '*.py' | head -20`.\n- Do not use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`: they only accept exact fully-qualified ids and usually return nothing.\n\n### 2. Implement the Solution Directly\n- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.\n- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.\n- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.\n\n### 3. Run Targeted Tests Only (Existing Tests May Be Broken)\n- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).\n- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.\n- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.\n- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.\n- If you need to locate the test file, find it explicitly with `find tests -name \"*<name>*.py\"` instead of running the test runner across the repo.\n\n### 4. Immediate Patch Submission\n- Once your targeted test passes:\n  1. Call `submit_patch` immediately.\n  2. Verify `patch_size > 0` and `files_changed > 0`.\n  3. Output a short summary of the fix to end the session.\n\n## Budget: 28 tool calls, 8 minutes\n- You have at most 28 tool calls and 8 minutes for this task. When either runs out, the working tree is graded as it is, so an edit made in time counts even if you never reach `submit_patch`.\n- Use at most about 10 tool calls to find and read the code.\n- By your 12th tool call you must have edited a source file. If you have not, stop exploring and make your best-guess edit in the most likely place now. Runs that are still only reading after 12 calls almost never succeed; more reading does not help.\n- After your edit: one compile check or one targeted test, a correction if needed, then `submit_patch`. Do not start new exploration after your 20th tool call.\n\n## Anti-Patterns to Avoid\n- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.\n- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) \u2014 always specify the exact test file path.\n- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** \u2014 your task is strictly to implement the fix for the reported issue in source code.\n- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.\n- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.\n- Do NOT refactor or reformat unrelated functions or files.\n- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.\n",
}

out = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
with zipfile.ZipFile(out / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, text in FILES.items():
        z.writestr(name, text)
with zipfile.ZipFile(out / "submission.zip") as z:
    assert z.testzip() is None and "agent.yaml" in z.namelist()
    print(f"submission.zip: {len(z.namelist())} files")
    for n in z.namelist():
        print("  ", n)
print(FILES["eval_config.yaml"])
